# N1 · Forward PINN — 1D cantilever
**Hands-on 1 · Slides 21–33 · planned 35 minutes.**

**Checkpoint sequence:** ① network (7 min), ② derivatives and PDE (9 min), ③ boundary conditions (9 min), ④ train and validate (10 min). The student version has a few short TODOs and self-checks. The instructor solution runs from top to bottom.

**What is known:** $E,I,L,q$ and support conditions. **Unknown:** the deflection function $w(x)$. No observations are required for this forward PINN.

**Option A — 1D cantilever | Cambridge PIML masterclass.**

A prismatic cantilever of length $L=3$ m, rectangular cross-section $b=0.125$ m, $h=0.20$ m and $E=210$ GPa carries a uniform distributed load $q=10$ kN/m. We take downward deflection as positive and use the Euler–Bernoulli convention

$$ EI\,w^{(4)}(x)=q, \quad w(0)=w'(0)=0,\quad w''(L)=w′′′(L)=0. $$

Its exact solution is

$$w(x)=\frac{q x^2}{24 EI}(6L^2-4Lx+x^2),\quad w(L)=\frac{qL^4}{8EI}=5.79\ \mathrm{mm}.$$

For numerical stability we use $\xi=x/L$ and **tip-deflection reference scaling** $u=w/w_\mathrm{ref}$ where $w_\mathrm{ref}=q_\mathrm{ref}L^4/(8EI)$ and $q_\mathrm{ref}=10$ kN/m. Then

$$\boxed{\frac{1}{8}\frac{d^4u}{d\xi^4}=\frac{q}{q_\mathrm{ref}}},\qquad u_\mathrm{exact}(\xi)=2\xi^2-\frac43\xi^3+\frac13\xi^4 \quad (q=q_\mathrm{ref}).$$

**Note on slides 23–26:** those slides use the alternative scaling $\hat u=EIw/(qL^4)$ so $\hat u^{(4)}=1$. Both are equivalent: $u=8\hat u$. This notebook uses $u$ so the expected tip value is 1 and loss terms have convenient scales. **Do not mix these dimensionless definitions in one equation.**

In [ ]:
import os
import math
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

os.makedirs('figures', exist_ok=True)
torch.set_default_dtype(torch.float64)  # higher derivatives are more stable in float64
try:
    torch.set_num_threads(2)
except RuntimeError:
    pass

def save_fig(name):
    plt.savefig('figures/' + name, dpi=190, bbox_inches='tight')
    plt.show()

def d_dx(y, x):
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

L = 3.0                        # m
E = 210e9                     # Pa
b, h = 0.125, 0.20            # m
I = b * h**3 / 12             # m^4
EI = E * I                    # N m^2
q_ref = 10e3                  # N/m
w_ref = q_ref * L**4 / (8*EI) # m

def u_exact(xi):
    return 2*xi**2 - (4/3)*xi**3 + xi**4/3

def m_exact(xi):
    # Cantilever internal bending-moment MAGNITUDE, kN m
    return q_ref * L**2 / 2 * (1-xi)**2 / 1000

print(f'EI = {EI:.3g} N m² | exact tip = {w_ref*1000:.3f} mm | root |M| = {q_ref*L**2/2000:.2f} kN m')

## Checkpoint 1 · Build a small smooth network
Use `tanh`, because we will differentiate four times. Input: one coordinate $\xi$. Output: one normalised displacement $u(\xi)$.

In [ ]:
# TODO 1: Create an nn.Sequential network with layers 1 → 32 → 32 → 32 → 1.
# Put nn.Tanh() between each hidden layer. Example: nn.Linear(1,32), nn.Tanh()
torch.manual_seed(12)
model = None  # TODO: replace with your network
assert isinstance(model, nn.Module), 'TODO 1: create the network first'
assert model(torch.tensor([[0.2],[0.8]])).shape == (2,1)
print('CHECK 1 PASSED: one input → one output')

## Checkpoint 2 · Differentiate four times
At interior **collocation points**, calculate the fourth derivative using AD. The strong-form residual is $r(\xi)=u^{(4)}(\xi)/8-1$.

In [ ]:
xi_f = torch.linspace(0,1,96).reshape(-1,1).requires_grad_(True)
u_f = model(xi_f)
# TODO 2: Calculate first, second, third and fourth derivatives using d_dx(y, xi_f).
# Hint: u_x = d_dx(u_f,xi_f)
u_x = None
u_xx = None
u_xxx = None
u_xxxx = None
assert u_xxxx is not None, 'TODO 2: complete the derivative ladder'
residual = u_xxxx/8 - 1.0
assert residual.shape == xi_f.shape and torch.isfinite(residual).all()
print('CHECK 2 PASSED: PDE residual calculated')

## Checkpoint 3 · Construct the full loss
The clamp enforces $u(0)=u'(0)=0$ and the free tip enforces $u''(1)=u′′′(1)=0$. These conditions make the fourth-order problem well posed. For clarity, we use **soft boundary losses**, not a hard-coded trial function.

In [ ]:
def pinn_losses(model, xi_f):
    u = model(xi_f)
    # TODO 3a: 4 derivatives, then loss_physics = mean((u4/8 - 1)**2)
    u1 = d_dx(u,xi_f)
    u2 = d_dx(u1,xi_f)
    u3 = d_dx(u2,xi_f)
    u4 = d_dx(u3,xi_f)
    loss_physics = None

    xi_0 = torch.tensor([[0.0]],requires_grad=True)
    xi_1 = torch.tensor([[1.0]],requires_grad=True)
    u0 = model(xi_0)
    du0 = d_dx(u0,xi_0)
    u1tip = model(xi_1)
    du1tip = d_dx(u1tip,xi_1)
    d2u1tip = d_dx(du1tip,xi_1)
    d3u1tip = d_dx(d2u1tip,xi_1)
    # TODO 3b: square and sum the four boundary quantities.
    loss_boundary = None
    assert loss_physics is not None and loss_boundary is not None, 'TODO 3: write both losses'
    return loss_physics, loss_boundary

loss_p, loss_b = pinn_losses(model,xi_f)
assert torch.isfinite(loss_p+loss_b)
print('CHECK 3 PASSED | PDE:',float(loss_p.detach()),'BC:',float(loss_b.detach()))

## Checkpoint 4 · Train and validate
**Training settings:** 2,400 Adam steps, 96 interior collocation points, boundary weight 5, default learning rate $2\times10^{-3}$ (lowered near the end). Training generally takes tens of seconds on a CPU, but timing depends on the Colab runtime.

In a teaching session, show the physics and boundary losses **separately**. A low total loss alone does not establish that equilibrium and supports are satisfied.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
history = {'epoch':[], 'physics':[], 'boundary':[], 'total':[]}

for epoch in range(2401):
    optimizer.zero_grad()
    loss_physics, loss_boundary = pinn_losses(model, xi_f)
    loss_total = loss_physics + 5.0*loss_boundary
    loss_total.backward()
    optimizer.step()

    if epoch == 1700:
        for group in optimizer.param_groups:
            group['lr'] = 0.0007
    if epoch % 100 == 0:
        history['epoch'].append(epoch)
        history['physics'].append(loss_physics.item())
        history['boundary'].append(loss_boundary.item())
        history['total'].append(loss_total.item())
    if epoch % 600 == 0:
        print(f'{epoch:4d} | physics={loss_physics.item():.2e} | boundary={loss_boundary.item():.2e}')
print('CHECK 4 PASSED: training complete')

In [ ]:
# Independent grid (not the 96 training collocation points)
xi_test = torch.linspace(0,1,250).reshape(-1,1).requires_grad_(True)
u_pred = model(xi_test)
du_pred = d_dx(u_pred,xi_test)
d2u_pred = d_dx(du_pred,xi_test)
d3u_pred = d_dx(d2u_pred,xi_test)
d4u_pred = d_dx(d3u_pred,xi_test)

w_pred_mm = (w_ref * u_pred.detach()*1000).numpy().ravel()
w_exact_mm = (w_ref * u_exact(xi_test.detach())*1000).numpy().ravel()
M_pred = (EI*w_ref/L**2*d2u_pred.detach()/1000).numpy().ravel()
M_exact = m_exact(xi_test.detach()).numpy().ravel()
x_axis = (L*xi_test.detach()).numpy().ravel()

err_tip_pct = abs(w_pred_mm[-1]-w_exact_mm[-1])/w_exact_mm[-1]*100
err_rmse_mm = np.sqrt(np.mean((w_pred_mm-w_exact_mm)**2))
print(f'Tip: {w_pred_mm[-1]:.3f} mm (exact {w_exact_mm[-1]:.3f} mm), error {err_tip_pct:.2f}%')
print(f'Root |M|: {M_pred[0]:.2f} kN·m (exact {M_exact[0]:.2f} kN·m)')
print(f'RMSE deflection: {err_rmse_mm:.4f} mm | test PDE RMSE: {torch.sqrt(torch.mean((d4u_pred/8-1)**2)).item():.3e}')

plt.figure(figsize=(7,4))
plt.plot(x_axis,w_exact_mm,'k',lw=2,label='Analytical')
plt.plot(x_axis,w_pred_mm,'--',lw=2,label='Forward PINN')
plt.xlabel('x (m)');plt.ylabel('Downward deflection (mm)');plt.title('Cantilever: forward deflection');plt.grid(alpha=.2);plt.legend();save_fig('N1_forward_deflection.png')

plt.figure(figsize=(7,4))
plt.plot(x_axis,M_exact,'k',lw=2,label='Analytical |M|')
plt.plot(x_axis,M_pred,'--',lw=2,label='PINN-derived EI w″')
plt.xlabel('x (m)');plt.ylabel('Bending-moment magnitude (kN·m)');plt.title('Cantilever: bending moment');plt.grid(alpha=.2);plt.legend();save_fig('N1_forward_moment.png')

plt.figure(figsize=(7,4))
plt.semilogy(history['epoch'],history['physics'],label='PDE loss')
plt.semilogy(history['epoch'],history['boundary'],label='Boundary loss')
plt.semilogy(history['epoch'],history['total'],label='Total loss',alpha=.7)
plt.xlabel('Adam epoch');plt.ylabel('Mean-square loss');plt.title('PINN training diagnostics');plt.grid(alpha=.2);plt.legend();save_fig('N1_training_history.png')

plt.figure(figsize=(7,4))
plt.plot(x_axis,(d4u_pred.detach()/8-1).numpy().ravel())
plt.xlabel('x (m)');plt.ylabel('Dimensionless PDE residual')
plt.title('Independent test grid: u⁽⁴⁾ / 8 − 1');plt.grid(alpha=.2);save_fig('N1_independent_residual.png')

## Optional 5-minute experiment — change **one** loss weight (Slide 33)
For a quick comparison, compare `bc_weight=5` with removing the boundary term (`bc_weight=0`) and train from identical random initial weights. This cell creates the actual `N1_loss_weight_experiment.png` plot. It is optional if time is tight.

In [ ]:
# Optional experiment: same architecture, seed and training budget

def quick_train(bc_weight):
    torch.manual_seed(33)
    m = nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,32),nn.Tanh(),nn.Linear(32,32),nn.Tanh(),nn.Linear(32,1))
    opt = torch.optim.Adam(m.parameters(),lr=0.002)
    x = torch.linspace(0,1,64).reshape(-1,1).requires_grad_(True)
    for ep in range(1200):
        opt.zero_grad()
        lp,lb = pinn_losses(m,x)
        (lp + bc_weight*lb).backward()
        opt.step()
    with torch.no_grad():
        pred_tip = m(torch.tensor([[1.0]])).item()
    return pred_tip, lp.item(), lb.item()

a = quick_train(5.0)
b = quick_train(0.0)
print('boundary weight 5   : tip u, PDE loss, BC loss =',a)
print('boundary weight 0    : tip u, PDE loss, BC loss =',b)
plt.figure(figsize=(6.7,3.7))
plt.bar(['BC weight = 5','No BC term'],[abs(a[0]-1),abs(b[0]-1)])
plt.ylabel('Absolute dimensionless tip-deflection error')
plt.title('Does less BC penalty improve the engineering answer?')
plt.grid(axis='y',alpha=.2);save_fig('N1_loss_weight_experiment.png')

### Files to paste into the PowerPoint
- **Slide 31:** `N1_forward_deflection.png`, `N1_forward_moment.png`, `N1_training_history.png`.
- **Slide 33:** `N1_loss_weight_experiment.png` if you run the optional experiment.
- Keep `N1_independent_residual.png` as an independent mechanics check.

**Validation reminder:** the signed internal moment for this downward-positive convention is often written $M=-EIw''$. We plot the **positive moment magnitude** $EIw''$, avoiding a sign-convention mismatch.

In [ ]:
# Optional: download all exported plots in one ZIP from Google Colab
import zipfile
with zipfile.ZipFile('plots_for_slides.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir('figures')):
        if name.endswith('.png'):
            z.write('figures/' + name, arcname=name)
print('Saved plots_for_slides.zip')
try:
    from google.colab import files
    files.download('plots_for_slides.zip')
except ImportError:
    print('Outside Colab: find the ZIP in the current notebook folder.')